# LexMind AI - Milestone 2: Embeddings & FAISS Retrieval

## Deep Learning Legal Intelligence Assistant

This notebook demonstrates:
1. Semantic embedding generation using Transformers
2. FAISS vector index creation
3. Similar case retrieval
4. Visualization of embeddings

**Model:** sentence-transformers/all-mpnet-base-v2 (768-dim embeddings)

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

## 2. Load Preprocessed Data

In [ ]:
# Load train and test data
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

print(f"Train documents: {len(train_df)}")
print(f"Test documents: {len(test_df)}")
print(f"Total documents: {len(train_df) + len(test_df)}")

train_df.head()

## 3. Load Generated Embeddings

In [ ]:
# Load embeddings
train_embeddings = np.load('../models/embeddings/train_embeddings.npy')
test_embeddings = np.load('../models/embeddings/test_embeddings.npy')

print(f"Train embeddings shape: {train_embeddings.shape}")
print(f"Test embeddings shape: {test_embeddings.shape}")
print(f"Embedding dimension: {train_embeddings.shape[1]}")
print(f"\nEmbedding statistics:")
print(f"  Mean: {train_embeddings.mean():.4f}")
print(f"  Std: {train_embeddings.std():.4f}")
print(f"  Min: {train_embeddings.min():.4f}")
print(f"  Max: {train_embeddings.max():.4f}")

## 4. Embedding Distribution Analysis

In [ ]:
# Plot embedding value distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(train_embeddings.flatten(), bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('Distribution of Embedding Values', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Embedding Value')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Box plot of first 20 dimensions
axes[1].boxplot([train_embeddings[:, i] for i in range(20)], showfliers=False)
axes[1].set_title('Embedding Values Across First 20 Dimensions', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Dimensionality Reduction - PCA Visualization

In [ ]:
# Apply PCA to reduce to 2D
print("Applying PCA (768D → 2D)...")
pca = PCA(n_components=2, random_state=42)
embeddings_2d_pca = pca.fit_transform(train_embeddings)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")

# Plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d_pca[:, 0], embeddings_2d_pca[:, 1], 
                     c=train_df['doc_type'].astype('category').cat.codes,
                     cmap='viridis', alpha=0.6, s=30)
plt.colorbar(scatter, label='Document Type')
plt.title('Legal Document Embeddings - PCA Visualization', fontsize=16, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Dimensionality Reduction - t-SNE Visualization

In [ ]:
# Apply t-SNE (on subset for speed)
print("Applying t-SNE (768D → 2D) on 500 samples...")
sample_size = min(500, len(train_embeddings))
sample_indices = np.random.choice(len(train_embeddings), sample_size, replace=False)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d_tsne = tsne.fit_transform(train_embeddings[sample_indices])

# Plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d_tsne[:, 0], embeddings_2d_tsne[:, 1],
                     c=train_df.iloc[sample_indices]['doc_type'].astype('category').cat.codes,
                     cmap='plasma', alpha=0.6, s=50)
plt.colorbar(scatter, label='Document Type')
plt.title('Legal Document Embeddings - t-SNE Visualization', fontsize=16, fontweight='bold')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Similar documents cluster together in embedding space")

## 7. Load FAISS Index and Test Retrieval

In [ ]:
from src.retrieval import FAISSRetriever
from src.embedder import LegalEmbedder

# Load retriever
print("Loading FAISS index...")
retriever = FAISSRetriever(embedding_dim=768)
retriever.load_index('../models/embeddings/train_embeddings.index')
retriever.set_documents(train_df)

print(f"✓ FAISS index loaded with {retriever.index.ntotal} vectors")

# Load embedder for query encoding
print("Loading embedding model...")
embedder = LegalEmbedder()
print("✓ Model loaded")

## 8. Test Query 1: Cheque Bounce Cases

In [ ]:
# Query 1
query_text = "What are precedents for cheque bounce under section 138?"
print(f"Query: {query_text}\n")

# Encode query
query_embedding = embedder.encode_texts([query_text])[0]

# Retrieve similar cases
results = retriever.retrieve(query_embedding, k=5)

print("Top 5 Similar Cases:\n")
for idx, row in results.iterrows():
    print(f"Rank {row['rank']}: {row['case_id']}")
    print(f"  Similarity Score: {row['similarity_score']:.4f}")
    print(f"  Type: {row['doc_type']}")
    print(f"  Text Preview: {row['text'][:150]}...")
    print()

## 9. Test Query 2: Contract Disputes

In [ ]:
# Query 2
query_text = "Cases related to breach of contract and damages"
print(f"Query: {query_text}\n")

query_embedding = embedder.encode_texts([query_text])[0]
results = retriever.retrieve(query_embedding, k=5)

print("Top 5 Similar Cases:\n")
for idx, row in results.iterrows():
    print(f"Rank {row['rank']}: {row['case_id']}")
    print(f"  Similarity Score: {row['similarity_score']:.4f}")
    print(f"  Type: {row['doc_type']}")
    print(f"  Text Preview: {row['text'][:150]}...")
    print()

## 10. Test Query 3: Criminal Law

In [ ]:
# Query 3
query_text = "Criminal cases involving theft and robbery"
print(f"Query: {query_text}\n")

query_embedding = embedder.encode_texts([query_text])[0]
results = retriever.retrieve(query_embedding, k=5)

print("Top 5 Similar Cases:\n")
for idx, row in results.iterrows():
    print(f"Rank {row['rank']}: {row['case_id']}")
    print(f"  Similarity Score: {row['similarity_score']:.4f}")
    print(f"  Type: {row['doc_type']}")
    print(f"  Text Preview: {row['text'][:150]}...")
    print()

## 11. Similarity Score Distribution

In [ ]:
# Test multiple queries and collect similarity scores
test_queries = [
    "cheque bounce section 138",
    "breach of contract",
    "criminal theft robbery",
    "property dispute",
    "divorce proceedings"
]

all_scores = []
for query in test_queries:
    query_emb = embedder.encode_texts([query])[0]
    results = retriever.retrieve(query_emb, k=10)
    all_scores.extend(results['similarity_score'].tolist())

# Plot distribution
plt.figure(figsize=(10, 6))
plt.hist(all_scores, bins=30, alpha=0.7, color='green', edgecolor='black')
plt.title('Distribution of Similarity Scores', fontsize=14, fontweight='bold')
plt.xlabel('Similarity Score')
plt.ylabel('Frequency')
plt.axvline(np.mean(all_scores), color='red', linestyle='--', label=f'Mean: {np.mean(all_scores):.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Similarity Score Statistics:")
print(f"  Mean: {np.mean(all_scores):.4f}")
print(f"  Median: {np.median(all_scores):.4f}")
print(f"  Std: {np.std(all_scores):.4f}")
print(f"  Min: {np.min(all_scores):.4f}")
print(f"  Max: {np.max(all_scores):.4f}")

## 12. Retrieval Performance Metrics

In [ ]:
import time

# Test retrieval speed
num_tests = 100
retrieval_times = []

print(f"Testing retrieval speed with {num_tests} queries...")
for i in range(num_tests):
    # Random query from test set
    random_idx = np.random.randint(0, len(test_embeddings))
    query_emb = test_embeddings[random_idx]
    
    start_time = time.time()
    results = retriever.retrieve(query_emb, k=5)
    end_time = time.time()
    
    retrieval_times.append((end_time - start_time) * 1000)  # Convert to ms

# Plot
plt.figure(figsize=(10, 6))
plt.hist(retrieval_times, bins=30, alpha=0.7, color='purple', edgecolor='black')
plt.title('FAISS Retrieval Speed Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Retrieval Time (ms)')
plt.ylabel('Frequency')
plt.axvline(np.mean(retrieval_times), color='red', linestyle='--', 
           label=f'Mean: {np.mean(retrieval_times):.2f} ms')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nRetrieval Performance:")
print(f"  Mean: {np.mean(retrieval_times):.2f} ms")
print(f"  Median: {np.median(retrieval_times):.2f} ms")
print(f"  95th percentile: {np.percentile(retrieval_times, 95):.2f} ms")
print(f"  Max: {np.max(retrieval_times):.2f} ms")
print(f"\n✓ Retrieval latency < 2 sec (PRD requirement met)")

## 13. Summary Statistics

In [ ]:
print("="*60)
print("MILESTONE 2 SUMMARY")
print("="*60)
print(f"\n📊 Dataset:")
print(f"  Training documents: {len(train_df):,}")
print(f"  Test documents: {len(test_df):,}")
print(f"  Total documents: {len(train_df) + len(test_df):,}")

print(f"\n🧠 Embeddings:")
print(f"  Model: sentence-transformers/all-mpnet-base-v2")
print(f"  Embedding dimension: {train_embeddings.shape[1]}")
print(f"  Train embeddings: {train_embeddings.shape}")
print(f"  Test embeddings: {test_embeddings.shape}")

print(f"\n🔍 FAISS Index:")
print(f"  Index type: IndexFlatL2")
print(f"  Vectors indexed: {retriever.index.ntotal:,}")
print(f"  Mean retrieval time: {np.mean(retrieval_times):.2f} ms")

print(f"\n✅ PRD Requirements Met:")
print(f"  ✓ Feature 3: Semantic Similar Case Retrieval")
print(f"  ✓ Feature 4: FAISS Vector Search Engine")
print(f"  ✓ Model A: Embedding Generator (all-mpnet-base-v2)")
print(f"  ✓ FR3: Generate embeddings for every case")
print(f"  ✓ FR4: Retrieve top-k similar precedents")
print(f"  ✓ NFR2: Top-k retrieval latency < 2 sec")

print(f"\n🎯 Next Steps:")
print(f"  → Milestone 3: Implement BART Summarization")
print(f"  → Milestone 4: Implement Legal BERT QA")
print(f"  → Milestone 5: Build Streamlit UI")
print("="*60)